# Embedding Fundamentals: Geometry Before Providers

| Field | Value |
|---|---|
| Stage | Embeddings and indexes |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
An embedding is useful only with a compatible similarity function, normalization policy, and task-specific evaluation.

## 30-Second Summary

This notebook builds small transparent vectors, compares dot product, cosine similarity, and Euclidean distance, and shows why vector magnitude can change rankings when vectors are not normalized.

## Why This Matters

Vector databases expose distance settings that look interchangeable. A mismatch between model output, normalization, and index metric can silently change retrieval order.

## Scope

| Covers | Does not cover |
|---|---|
| Vector dimensions, dot/cosine/L2, normalization, ranking checks | Neural training, ANN internals, hosted-model benchmarking |


## Mental Model

```text
text -> embedding model/version -> vector -> optional normalization -> similarity metric -> ranking
```


In [1]:
from math import sqrt

def dot(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=True))

def norm(vector: list[float]) -> float:
    return sqrt(dot(vector, vector))

def normalize(vector: list[float]) -> list[float]:
    length = norm(vector)
    if length == 0: raise ValueError("Cannot normalize a zero vector.")
    return [value / length for value in vector]

def cosine(left: list[float], right: list[float]) -> float:
    return dot(left, right) / (norm(left) * norm(right))

def l2(left: list[float], right: list[float]) -> float:
    return sqrt(sum((a - b) ** 2 for a, b in zip(left, right, strict=True)))


## How It Works

Dot product rewards alignment and magnitude. Cosine divides out magnitude and compares direction. Euclidean distance measures straight-line separation. On L2-normalized vectors, cosine and Euclidean rankings are monotonically related; without normalization they can disagree.


## Baseline

The baseline ranks raw vectors by dot product without documenting magnitude. A long vector aligned with the query can dominate a closer-direction vector.


In [2]:
query = [1.0, 0.0]
candidates = {
    "same_direction_large": [10.0, 0.0],
    "close_direction": [0.9, 0.1],
    "orthogonal_large": [0.0, 20.0],
}
raw_dot_ranking = sorted(candidates, key=lambda name: dot(query, candidates[name]), reverse=True)
[(name, dot(query, candidates[name])) for name in raw_dot_ranking]


[('same_direction_large', 10.0),
 ('close_direction', 0.9),
 ('orthogonal_large', 0.0)]

## Technique Implementation

We compute all three measures explicitly and repeat dot-product ranking after normalization. Every compared vector has the same dimension; zero vectors are rejected rather than converted into misleading similarities.


In [3]:
comparison = {
    name: {
        "dot": round(dot(query, vector), 4),
        "cosine": round(cosine(query, vector), 4),
        "l2": round(l2(query, vector), 4),
        "magnitude": round(norm(vector), 4),
    }
    for name, vector in candidates.items()
}
comparison


{'same_direction_large': {'dot': 10.0,
  'cosine': 1.0,
  'l2': 9.0,
  'magnitude': 10.0},
 'close_direction': {'dot': 0.9,
  'cosine': 0.9939,
  'l2': 0.1414,
  'magnitude': 0.9055},
 'orthogonal_large': {'dot': 0.0,
  'cosine': 0.0,
  'l2': 20.025,
  'magnitude': 20.0}}

## Controlled Experiment

The controlled change is normalization. We compare raw dot ranking with normalized dot, cosine, and ascending L2 rankings on the same vectors.


In [4]:
normalized_query = normalize(query)
normalized_candidates = {name: normalize(vector) for name, vector in candidates.items()}
rankings = {
    "raw_dot": raw_dot_ranking,
    "normalized_dot": sorted(candidates, key=lambda name: dot(normalized_query, normalized_candidates[name]), reverse=True),
    "cosine": sorted(candidates, key=lambda name: cosine(query, candidates[name]), reverse=True),
    "normalized_l2": sorted(candidates, key=lambda name: l2(normalized_query, normalized_candidates[name])),
}
rankings


{'raw_dot': ['same_direction_large', 'close_direction', 'orthogonal_large'],
 'normalized_dot': ['same_direction_large',
  'close_direction',
  'orthogonal_large'],
 'cosine': ['same_direction_large', 'close_direction', 'orthogonal_large'],
 'normalized_l2': ['same_direction_large',
  'close_direction',
  'orthogonal_large']}

## Evaluation

Raw dot product puts the large same-direction vector first because magnitude is part of the score. After normalization, dot, cosine, and L2 agree on the full order. This geometric check does not establish semantic quality; that requires labeled queries and documents.


In [5]:
assert rankings["normalized_dot"] == rankings["cosine"] == rankings["normalized_l2"]
assert abs(cosine([1, 0], [10, 0]) - 1.0) < 1e-12
assert abs(l2(normalize([1, 0]), normalize([0, 1])) - sqrt(2)) < 1e-12
try:
    cosine([1.0], [1.0, 2.0])
except ValueError:
    pass
else:
    raise AssertionError("Dimension mismatch must fail.")
print("Embedding geometry checks passed.")


Embedding geometry checks passed.


## Decision Guide

| Situation | Metric/policy |
|---|---|
| Model recommends cosine | Normalize or use cosine explicitly |
| Magnitude carries meaning | Dot product, documented and tested |
| Euclidean-trained representation | L2 with compatible normalization |
| Unknown provider behavior | Inspect docs and verify rankings locally |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Ranking changes after migration | Metric/normalization changed | Log model, dimension, metric, normalization |
| All scores near zero | Model/query domain mismatch | Evaluate labeled domain queries |
| Insert/query dimension error | Mixed models or versions | Version indexes and reject mismatches |
| Perfect demo scores | Tiny/easy labels | Expand and segment the golden set |


## Production Notes

### Observability
Record model/version, dimension, normalization, metric, score distribution, and zero-vector rate.

### Safety and Guardrails
Embeddings can leak information; apply authorization before retrieval and protect vectors like derived sensitive data.

### Latency and Cost
Batch documents, cache by content hash and model version, and measure query and indexing paths separately.


## Practice

Add a candidate whose direction is slightly worse but magnitude is 100× larger. Predict each ranking before running it.

## Recall

Toggle - Recall: When do cosine and L2 rankings agree?
For the same L2-normalized vectors.

Toggle - Recall: Does a high cosine score prove relevance?
No; relevance depends on the model, domain, labels, and task.

## Sources

- [Sentence Transformers: similarity metrics](https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html)
- [scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the demonstrated geometry | Add ANN recall and domain-mismatch experiments |
